<a href="https://colab.research.google.com/github/zsgwu/G5_GWU_CAPST/blob/main/notebooks/03_rag_query_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EduYou — RAG Query Notebook (Azure OpenAI)

**File:** `rag_query_eduyou.ipynb`  

This notebook mirrors the professor’s `07_RAG_query.ipynb` template and is adapted for the EduYou capstone.

It:
1. Loads precomputed document embeddings
2. Embeds a user query using the **same embedding model**
3. Computes cosine similarity
4. Retrieves Top‑K documents
5. (Optional) passes retrieved text to an LLM

---

## Required inputs
- Document table: `cleaned/eduyou_cip_docs_for_embedding.csv`
- Embeddings table: `embeddings/eduyou_embeddings_<MODEL>.csv`

> ⚠️ Both must be generated using the **same embedding model**.


In [ ]:
import os
import numpy as np
import pandas as pd
from openai import AzureOpenAI
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1) Configuration

Ensure the same Azure OpenAI embedding deployment is used here as in `get_embeddings_eduyou.ipynb`.


In [ ]:
import os

# Azure OpenAI configuration
AZURE_OPENAI_ENDPOINT = os.getenv(
    'AZURE_OPENAI_ENDPOINT',
    'https://gw-sb-aoai-05.openai.azure.com/'
)

AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')

# ✅ Match professor + embeddings notebook
AZURE_OPENAI_API_VERSION = '2025-04-01-preview'

# ✅ Must match embeddings notebook exactly
AZURE_OPENAI_DEPLOYMENT_ID = os.getenv(
    'AZURE_OPENAI_DEPLOYMENT_ID',
    'text-embedding-3-small'
)

# File paths
DOCS_CSV = '/content/drive/MyDrive/group-5/RAG_data/cleaned/eduyou_cip_docs_for_embedding.csv'
EMB_CSV = '/content/drive/MyDrive/group-5/RAG_data/embeddings/eduyou_embeddings_text-embedding-3-small.csv'

# Retrieval parameters
TOP_K = 5

In [ ]:
print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("API version:", AZURE_OPENAI_API_VERSION)
print("Deployment:", AZURE_OPENAI_DEPLOYMENT_ID)
print("Embeddings file:", EMB_CSV)

Endpoint: https://gw-sb-aoai-05.openai.azure.com/
API version: 2025-04-01-preview
Deployment: text-embedding-3-small
Embeddings file: /content/drive/MyDrive/group-5/RAG_data/embeddings/eduyou_embeddings_text-embedding-3-small.csv


## 2) Load documents and embeddings


In [ ]:
# Load source documents and embeddings
docs = pd.read_csv(DOCS_CSV, low_memory=False)
embs = pd.read_csv(EMB_CSV, low_memory=False)

# Ensure alignment via doc_id
docs['doc_id'] = docs['doc_id'].astype(str)
embs['doc_id'] = embs['doc_id'].astype(str)

missing_embs = set(docs['doc_id']) - set(embs['doc_id'])
assert len(missing_embs) == 0, (
    f"Missing embeddings for {len(missing_embs)} documents"
)

# Extract embedding matrix
embed_cols = [c for c in embs.columns if c.startswith('dim_')]
X = embs[embed_cols].astype(float).values

print('Documents:', len(docs))
print('Embeddings loaded:', len(embs))
print('Embedding dimension:', X.shape[1])

Documents: 1312
Embeddings loaded: 1312
Embedding dimension: 1536


## 3) Embed a user query


In [ ]:
import os
import getpass

if not os.getenv("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("AZURE_OPENAI_API_KEY: ")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

AZURE_OPENAI_API_KEY: ··········


In [ ]:
print(AZURE_OPENAI_API_KEY is not None)

True


In [ ]:
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# User query
query = "What careers and salaries are associated with Communication and Media Studies?"

# Embed query using the SAME client and deployment as documents
q_resp = client.embeddings.create(
    model=AZURE_OPENAI_DEPLOYMENT_ID,
    input=query
)

# Convert to numpy array (1 x embedding_dim)
q_vec = np.array(q_resp.data[0].embedding, dtype=float).reshape(1, -1)

print("Query embedded.")
print("Query embedding shape:", q_vec.shape)


Query embedded.
Query embedding shape: (1, 1536)


## 4) Retrieve Top‑K documents by cosine similarity


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute cosine similarity (documents x query)
sims = cosine_similarity(X, q_vec).flatten()

# Get indices of top-K most similar documents
top_idx = np.argsort(sims)[-TOP_K:][::-1]

# Select top-K embeddings
top_embs = embs.iloc[top_idx].copy()
top_embs['similarity'] = sims[top_idx]

# Join back to documents for full metadata/text
results = docs.merge(
    top_embs[['doc_id', 'similarity']],
    on='doc_id',
    how='inner'
).sort_values('similarity', ascending=False)

# Display key fields
results[['doc_id', 'cip_title', 'degree_level', 'similarity']]

,doc_id,cip_title,degree_level,similarity
1,1001_Masters_Degree,Communications Technologies/Technicians.,Master's Degree,0.504527
0,1001_Associates_Degree,Communications Technologies/Technicians.,Associate's Degree,0.483175
4,5205_Masters_Degree,Business/Corporate Communications.,Master's Degree,0.474672
3,5205_Bachelors_Degree,Business/Corporate Communications.,Bachelor's Degree,0.469054
2,5102_Masters_Degree,Communication Disorders Sciences and Services.,Master's Degree,0.464480


## 5) Inspect retrieved document text


In [ ]:
for i, row in results.head(TOP_K).iterrows():
    print("=" * 80)
    print(f"Doc ID: {row['doc_id']}")
    print(f"Similarity score: {row['similarity']:.3f}")
    print("-" * 80)
    print(str(row.get('text', ''))[:1200])
    print()
    print(f"CIP title: {row.get('cip_title', '')}")
    print(f"Degree level: {row.get('degree_level', '')}")
    print()

Doc ID: 1001_Masters_Degree
Similarity score: 0.505
--------------------------------------------------------------------------------
CIP family (cip4): 1001
Field of study: Communications Technologies/Technicians.
Degree level: Master's Degree
Median earnings (4 years after completion, national): $71,506

Related occupations with OEWS wages (top 4 by employment):
- 27-4032 Film and Video Editors | median wage $70,980 | employment 28,860
- 27-3099 Media and Communication Workers, All Other | median wage $71,770 | employment 23,590
- 27-4012 Broadcast Technicians | median wage $53,920 | employment 21,080
- 27-4014 Sound Engineering Technicians | median wage $66,430 | employment 13,050

CIP title: Communications Technologies/Technicians.
Degree level: Master's Degree

Doc ID: 1001_Associates_Degree
Similarity score: 0.483
--------------------------------------------------------------------------------
CIP family (cip4): 1001
Field of study: Communications Technologies/Technicians.
Degree 

## 6) (Optional) Pass retrieved context to an LLM with DATASET-LEVEL citations

This mirrors the professor’s RAG pattern: retrieved documents are concatenated and provided as context to a generation model.


In [ ]:
# Combine retrieved document text into context
context = "\n\n".join(
    results.head(TOP_K)['text'].astype(str).tolist()
)

# Define dataset sources (for citation)
DATA_SOURCES = (
    "U.S. Department of Education College Scorecard / IPEDS; "
    "U.S. Bureau of Labor Statistics Occupational Employment and Wage Statistics (OEWS)"
)

# Build prompt
prompt = f"""
You are an educational assistant.

Using ONLY the information provided in the context below,
answer the user's question clearly and accurately.

When presenting facts (e.g., careers, earnings),
cite the underlying data source using:
[Source: {DATA_SOURCES}]

Context:
{context}

Question:
{query}

Answer (with source citations):
"""

response = client.chat.completions.create(
    model="gpt-5.1",  # must match deployed chat model
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Here are careers and earnings associated with the communication‑related fields listed in your data:

1. **Communications Technologies/Technicians (Master’s Degree)**  
   - Median earnings 4 years after completion: **$71,506**  
   - Related occupations:  
     - **Film and Video Editors** – median wage **$70,980**; employment **28,860**  
     - **Media and Communication Workers, All Other** – median wage **$71,770**; employment **23,590**  
     - **Broadcast Technicians** – median wage **$53,920**; employment **21,080**  
     - **Sound Engineering Technicians** – median wage **$66,430**; employment **13,050**  
   [Source: U.S. Department of Education College Scorecard / IPEDS; U.S. Bureau of Labor Statistics Occupational Employment and Wage Statistics (OEWS)]

2. **Communications Technologies/Technicians (Associate’s Degree)**  
   - Median earnings 4 years after completion: **$35,292**  
   - Related occupations:  
     - **Film and Video Editors** – median wage **$70,980**; empl

## Summary

- ✅ One row = one document
- ✅ Same embedding model for docs and query
- ✅ Cosine similarity retrieval
- ✅ Added citations
